# Lab 4.2 &mdash; Reading a Model's Confidence (softmax over real logits)

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Run a real model to get its raw output scores (logits)
- Convert logits into probabilities with softmax you implement
- Read the predicted label and its confidence

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; GPT generation / BERT sentiment](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-04-02")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
A classifier's final layer outputs raw scores (**logits**), one per class. **Softmax** turns them
into probabilities that sum to 1; the **argmax** is the prediction and the **max probability** is the
**confidence**. The `pipeline` does this for you &mdash; here we pull the **real logits** out of
distilbert-SST-2 ourselves and run softmax by hand, so you see exactly where that confidence score
comes from.

## Build it
Implement softmax, then turn real logits into (label, confidence).

In [ ]:
import numpy as np
def softmax(x):
    x = np.asarray(x, dtype=float)
    e = ___   # TODO: np.exp(x - x.max())  (subtract max for numerical stability)
    return ___   # TODO: e divided by its sum

def label_and_confidence(logits):
    p = softmax(logits)
    label = ___   # TODO: index of the largest probability (int(np.argmax(p)))
    conf = ___    # TODO: the largest probability (float(p.max()))
    return label, conf

## Run it for real
Get REAL logits from the model, then apply your softmax.

In [ ]:
try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    name = "distilbert-base-uncased-finetuned-sst-2-english"
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name); model.eval()
    print("class order (id -> label):", model.config.id2label)

    for text in ["a brilliant and moving masterpiece", "not bad at all", "it was fine i guess"]:
        enc = tok(text, return_tensors="pt")
        with torch.no_grad():
            logits = model(**enc).logits[0].numpy()   # the REAL raw scores [neg, pos]
        label, conf = label_and_confidence(logits)
        print(f"{text!r}")
        print(f"   logits={np.round(logits,3)}  ->  label={model.config.id2label[label]}  confidence={conf:.3f}")
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- The **logits are real** model output &mdash; two numbers, one per class. Softmax turns them into a probability you can read as confidence.
- This SST-2 model is **very confident** even on negation (`"not bad at all"` &rarr; POSITIVE, correctly) &mdash; strong fine-tuned models often are. Pushing the confidence toward 0.5 takes genuinely ambiguous or off-topic text (that is your **Your turn**).
- This confidence is *exactly* the `score` the `pipeline` returned in Lab 4.1 &mdash; you just reproduced its internals.

## Your turn (open task &mdash; no grader)
Find sentences that make the model **unsure** (confidence closer to 0.5) &mdash; negation,
mixed sentiment, or off-topic text. Confidence is vital when a human reviews the model's output: a
"good" answer names a threshold below which you would route a prediction to a human, and one real
sentence that falls under it.

---
### Key takeaway
Softmax over real logits *is* the confidence score. It tells you *how sure* the model is &mdash; the number a human-in-the-loop system watches.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>